In [193]:
import numpy as np
import pandas as pd
import string
from collections import defaultdict, Counter
import random

In [194]:
df = pd.read_csv(r"PersianTwitterDataset.csv")
df.head()

,Unnamed: 0,Tweets,Numeric Labels,Label
0,0,احساس می‌کنم غایت زندگی من اینه که یکی یروز بف...,4,Intense Emotions
1,1,ای بابا نصیب هرکسی نمیشه که، فقط دعای خیر پدر ...,0,Happy
2,2,با من مثل بقیه رفتار نکن آشغال,2,Angry
3,3,من علاقه شدیدی به شنیدن «بغلم کن» از طرف اونی ...,4,Intense Emotions
4,4,چیز کیک گرم و نرم میخوام.,0,Happy


In [197]:
def Normalize_persian(x):
    x = x.replace("\u200c", " ")
    x = x.replace("\u200f", " ")
    for c in string.punctuation:
        x = x.replace(c, " ")

    return x.split()

text = "من خوش قلب هستم!"
print(Normalize_persian(text))

['من', 'خوش', 'قلب', 'هستم']


In [198]:
class ngram_predictor:
    def __init__(self, n, alpha=1, random_state=26):
        self.n = n
        self.alpha = alpha
        self.random_state = random_state

    def fit(self, df):
        self.df = df
        self.grams = []
        self.words = []
        texts = [Normalize_persian(self.df["Tweets"].iloc[idx]) for idx in range(self.df.shape[0])]

        for text in texts:
            for i in range(len(text)):
                self.words.append(text[i])
                if i <= len(text) - self.n:
                    gram = tuple(text[j] for j in range(i, i+self.n))
                    self.grams.append(gram)

        self.grams_counts = Counter(self.grams)

        return self

    def predict_next(self, x, return_scores=False):
        x_norm = Normalize_persian(x)
        scores = defaultdict()
        nm1_gram = ngram_predictor(n=self.n-1)
        nm1_gram.fit(self.df)
        self.nm1_counts = nm1_gram.grams_counts

        for word in self.words:
            new_gram = tuple(x_norm[-(self.n-1):] + [word])
            scores[word] = (self.grams_counts[new_gram] + self.alpha) / (self.nm1_counts[tuple(x_norm[-(self.n-1):])] + self.alpha * len(self.words))

        if return_scores:
            return max(set(scores), key=scores.get), scores    
        return max(set(scores), key=scores.get)

    def generate_sentence(self, seq, max_length=12):
        rng = random.Random(self.random_state)
        x_norm = Normalize_persian(seq)
        while len(x_norm) < max_length:
            _, scores = self.predict_next(" ".join(x_norm), return_scores=True)
            next_word = rng.choices(list(scores.keys()), weights=list(scores.values()))
            x_norm = x_norm + next_word

        return " ".join(x_norm + ["."])

ngram_model = ngram_predictor(n=2)
ngram_model.fit(df)

ngram_model.generate_sentence("مملکت نیست که")

'مملکت نیست که آدابی والا شد😂 نکنم وجود آزار اصفهونی جالب کتکم .'